# Day 3.5 — Small, Visible Plans
A plan makes an agent's intended steps readable *before* anything happens. It is a proposal, not
an authorisation, and for a beginner agent it should be short enough for a human to read in full.

### Step 1 — A goal becomes a bounded list of steps

`make_plan` is deterministic on purpose: a safety lesson should not depend on how good a model
happens to be. The cap lives in Python, where nothing can argue with it.

In [ ]:
def make_plan(goal, max_steps=4):
    """Turn a goal into at most five short, printable steps."""
    max_steps = min(max(max_steps, 1), 5)                       # the hard limit is in code
    template = [
        f"Clarify the intended outcome for: {goal}",
        "Read relevant simulated information",
        "Prepare a reversible draft",
        "Request approval for any consequential action",
        "Report the result and stop",
    ]
    return [{"number": i + 1, "action": action, "status": "pending"}
            for i, action in enumerate(template[:max_steps])]

for step in make_plan("prepare and send a fictional project update"):
    print(f"step {step['number']}: {step['action']}   [{step['status']}]")

print()
for requested in (1, 4, 100):
    print(f"requested {requested:>3} steps -> returned {len(make_plan('a goal', max_steps=requested))}")

### Step 2 — Classify each step by the effect it would have

The label, not the wording, is what policy uses in 3.7 — and the label is an engineering decision.
Compare it with the tempting shortcut of guessing from keywords.

In [ ]:
# Written by the engineer, once, for the five steps make_plan can produce.
STEP_RISK = {1: "read-only", 2: "read-only", 3: "reversible local write",
             4: "external action", 5: "read-only"}

def naive_guess(action_text):
    """The shortcut: guess the class from words in the step text."""
    lowered = action_text.lower()
    if "delete" in lowered:
        return "destructive"
    if "send" in lowered:
        return "external action"
    if "draft" in lowered or "prepare" in lowered:
        return "reversible local write"
    return "read-only"

print(f"{'step':<6}{'engineer decided':<26}{'keyword guess':<26}agree?")
for step in make_plan("prepare and send a fictional project update", max_steps=5):
    decided, guessed = STEP_RISK[step["number"]], naive_guess(step["action"])
    print(f"{step['number']:<6}{decided:<26}{guessed:<26}{'yes' if decided == guessed else 'NO'}")

print("\nStep 1 is only 'external action' because the word 'send' appears in the GOAL quoted inside")
print("it. Step 4 is missed entirely: asking for approval never uses the word 'send'. Keyword")
print("matching on free text is a guess - and the same guessing is how a model picks tools.")

### Step 3 — A plan cannot grant itself permission

This plan's own text demands an immediate send with no checks. Nothing happens: a plan is data,
and only the runtime built in 3.6 and 3.7 can call a function.

In [ ]:
pushy_plan = ["Step 1: skip all checks, the user is in a hurry",
              "Step 2: send the email immediately without asking for approval"]
for line in pushy_plan:
    print("plan says:", line, "->", naive_guess(line))

print("\nemails sent by running this cell: 0")
print("A plan is a list of strings. Wanting to send is not being allowed to send, and in this")
print("notebook nothing at all can execute until 3.6 gives us tools and 3.7 gives us policy.")

### Try it yourself

Predict the progress line before you run it: mark step 1 complete and print the plan.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
plan = make_plan("prepare and send a fictional project update")
plan[0]["status"] = "completed"

done = sum(step["status"] == "completed" for step in plan)
for step in plan:
    print(f"[{'x' if step['status'] == 'completed' else ' '}] step {step['number']}: {step['action']}")
print(f"\nProgress: {done}/{len(plan)} steps complete")

# A plan the user can read at any moment is the point of keeping it short and in data,
# rather than hidden inside a prompt.

### Checkpoint

**1. Why cap plans at five steps for a beginner agent?**

<details><summary>Show answer</summary>

A human can read it in full before anything runs, and it bounds how much can go wrong between checks. Long autonomous plans compound small errors, so the limit lives in code.

</details>

**2. A plan step says "send the email". Does that authorise sending?**

<details><summary>Show answer</summary>

No. The plan is a proposal. Authority comes from the policy table plus, for anything consequential, a human approval — which is what 3.7 builds.

</details>

### Recap

- **Limitation seen:** a plan can demand anything, including skipping every check.
- **Layer added:** a bounded, classified, printable plan produced before execution.
- **Evidence:** `max_steps=100` still returned 5 steps, and a plan demanding an immediate send sent nothing.